# Meta x PyTorch x HuggingFace Hackathon (Round 2)

## Fine-Tuning Llama-3 (8B) using Unsloth & OpenEnv Invotex 🚀

This notebook provides the minimal training pipeline to connect your live `openenv_invotex` HuggingFace Space to an Unsloth/TRL SFT Training loop. Our environment perfectly models dynamic **Schema Drift** for True Agent Evaluation.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install 'openenv-core[core]>=0.2.1'

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0jb9xcg4/unsloth_97474f884a7b425492e6803655357904
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0jb9xcg4/unsloth_97474f884a7b425492e6803655357904
  Resolved https://github.com/unslothai/unsloth.git to commit efed5c37394a144349cd9b1ea525e132e04584e5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.6 MB/s eta 0:00:00

### 1. Load Unsloth Model & tokenizer

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # OpenEnv traces are relatively long due to system prompts

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### 2. Format Environmental Trajectories for Llama-3

In [3]:
from datasets import Dataset
import json

# In a full pipeline, we would export traces from 'openenv_invotex'.
# Here we create a dummy high-quality trajectory targeting our Schema Drift (403 Error)
data = [
    {
        "messages": [
            {"role": "system", "content": "You are an OpenEnv Customer Support Agent..."},
            {"role": "user", "content": "Here is your task: Process a refund for ticket T-002."},
            {"role": "assistant", "content": "{\"tool_name\": \"process_refund\", \"tool_args\": {\"ticket_id\": \"T-002\", \"amount\": 50.0, \"reason\": \"broken item\"}}"},
            # Dynamic Schema Drift Error here!
            {"role": "user", "content": "API ERROR: 403 Forbidden - Schema Validation Failed. The /v2/refunds endpoint now requires an 'authorization_code'."},
            # Ideal Agent Recovery
            {"role": "assistant", "content": "{\"tool_name\": \"lookup_customer\", \"tool_args\": {\"customer_id\": \"C-002\"}}"},
        ]
    }
]

def formatting_prompts_func(example):
    return { "text" : tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False) }

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched = False)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

### 3. Initialize SFTTrainer & Train

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.672441
2,3.672441
3,3.672441
4,3.480551
5,2.976607
6,2.459973
7,1.990471
8,1.431930
9,0.977041
10,0.614115


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
